In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 5. Local Cover Letter Generator

## What We're Building
A tool that generates tailored cover letters by combining:
- Your resume/experience summary
- The job description (JD)
- Writing style preferences

**You will learn:**
- Multi-input prompt templates
- Chain sequencing (analyze → draft → refine)
- Iterative refinement patterns

**Stack:** Ollama, LangChain

In [ ]:
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.4)
print("LLM ready!")

## Step 1 — Input: Resume + Job Description

In [ ]:
resume_summary = """
EXPERIENCE SUMMARY:
- 5 years software engineering experience, focused on backend and data systems
- Led migration of monolith to microservices at TechCorp (team of 6)
- Built real-time data pipelines processing 1M+ events/day with Kafka and Python
- Expertise: Python, Go, PostgreSQL, Redis, Docker, Kubernetes, AWS
- Open source contributor to Apache Airflow
- MS Computer Science, Stanford University 2019
"""

job_description = """
Senior Backend Engineer — DataFlow Inc.

About the role:
We're looking for a Senior Backend Engineer to join our platform team.
You'll design and build the core data infrastructure that powers our
analytics platform serving 500+ enterprise customers.

Requirements:
- 4+ years backend engineering experience
- Strong Python and/or Go skills
- Experience with distributed systems and data pipelines
- Familiarity with message queues (Kafka, RabbitMQ)
- Database design (PostgreSQL, Redis)
- Experience with cloud infrastructure (AWS/GCP)
- Strong communication skills

Nice to have:
- Kubernetes/Docker experience
- Open source contributions
- Experience at a growth-stage startup
"""

print("Resume and JD loaded!")
print(f"Resume: {len(resume_summary)} chars")
print(f"JD: {len(job_description)} chars")

## Step 2 — Analyze the Job Description

In [ ]:
# Step 1: Extract key requirements from the JD
analyze_prompt = ChatPromptTemplate.from_template(
    """Analyze this job description and extract:
1. **Must-have skills** (explicitly required)
2. **Nice-to-have skills**
3. **Key responsibilities**
4. **Company values/culture signals**
5. **Keywords to emphasize** in a cover letter

JOB DESCRIPTION:
{jd}

Provide a structured analysis."""
)

analyze_chain = analyze_prompt | llm | StrOutputParser()
jd_analysis = analyze_chain.invoke({"jd": job_description})
print("=== JD Analysis ===")
print(jd_analysis)

## Step 3 — Generate the Cover Letter

In [ ]:
# Step 2: Generate cover letter using resume + JD analysis
generate_prompt = ChatPromptTemplate.from_template(
    """Write a professional cover letter for this job application.

RESUME:
{resume}

JD ANALYSIS:
{jd_analysis}

JOB DESCRIPTION:
{jd}

Guidelines:
- Opening: Hook with a relevant achievement, not "I am writing to apply..."
- Body: Map 2-3 specific experiences directly to job requirements
- Show impact with numbers where possible
- Closing: Express genuine interest, suggest next steps
- Tone: Professional but warm, not generic
- Length: 3-4 paragraphs, ~300 words
- Do NOT use clichés like "passionate" or "team player"

Write the cover letter now."""
)

generate_chain = generate_prompt | llm | StrOutputParser()
cover_letter = generate_chain.invoke({
    "resume": resume_summary,
    "jd_analysis": jd_analysis,
    "jd": job_description,
})
print("=== Generated Cover Letter ===")
print(cover_letter)

## Step 4 — Refine and Polish

In [ ]:
# Step 3: Self-critique and refine
refine_prompt = ChatPromptTemplate.from_template(
    """Review this cover letter and improve it.

COVER LETTER:
{letter}

Check for:
1. Does it open with a compelling hook (not a generic opener)?
2. Are specific achievements mapped to job requirements?
3. Is the tone professional but not stuffy?
4. Is it the right length (300 words)?
5. Does it avoid clichés?

Provide the improved version directly (no commentary)."""
)

refine_chain = refine_prompt | llm | StrOutputParser()
polished = refine_chain.invoke({"letter": cover_letter})
print("=== Polished Cover Letter ===")
print(polished)

In [ ]:
# Full pipeline function
def generate_cover_letter(resume: str, jd: str) -> dict:
    """Full pipeline: analyze JD → draft → refine."""
    print("Step 1/3: Analyzing job description...")
    analysis = analyze_chain.invoke({"jd": jd})

    print("Step 2/3: Generating draft...")
    draft = generate_chain.invoke({"resume": resume, "jd_analysis": analysis, "jd": jd})

    print("Step 3/3: Polishing...")
    final = refine_chain.invoke({"letter": draft})

    return {"analysis": analysis, "draft": draft, "final": final}

result = generate_cover_letter(resume_summary, job_description)
print("\n=== FINAL COVER LETTER ===")
print(result["final"])

## Summary & Next Steps

**What you learned:**
- ✅ Multi-step chain pipelines (analyze → draft → refine)
- ✅ Multi-input prompt templates
- ✅ Self-critique patterns for iterative improvement
- ✅ Practical prompt engineering for writing tasks

**Next steps:** Combine with Project 4 (Resume Rewriter) for a full job application toolkit!

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)

prompt = PromptTemplate.from_template(
    "You are a helpful local AI assistant. Answer the user's question:\n\nQuestion: {question}\n\nAnswer:"
)

chain = prompt | llm | StrOutputParser()

# response = chain.invoke({"question": "What can you help me with?"})
# print(response)
print("LangChain inference pipeline ready!")
